# **Semantic Web & Linked Open Data with Python**
## บทที่ 10 — Ontology Design

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch10/ch10_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://oops.linkeddata.es/
*   https://ontologydesignpatterns.org/

📖 เอกสารประกอบบทนี้: [`10-ontology-design/`](https://github.com/Rathachai/semantic-web/tree/main/10-ontology-design)

---

### Setup

In [1]:
!pip install -q rdflib

In [2]:
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import OWL, RDF, RDFS

EX = Namespace('http://example.org/lib#')

## Competency Question → SPARQL

การออกแบบเริ่มจากคำถามที่ ontology ต้องตอบได้ เช่น:
> *"หนังสือเล่มไหนบ้างที่ยังไม่ถูกคืน?"*

แปลงเป็น SPARQL แล้วใช้เป็นแบบทดสอบ (test) ของ ontology

In [3]:
ttl = """
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex:   <http://example.org/lib#> .

ex:loan001 ex:borrowedBook ex:book1 ; ex:returned false .
ex:loan002 ex:borrowedBook ex:book2 ; ex:returned true .
ex:loan003 ex:borrowedBook ex:book3 ; ex:returned false .
"""
g = Graph().parse(data=ttl, format="turtle")

cq1 = "SELECT ?book WHERE { ?loan ex:borrowedBook ?book ; ex:returned false . }"
res = g.query(cq1, initNs={"ex": EX})
print("หนังสือที่ยังไม่คืน:")
for row in res:
    print(" ", row.book)

หนังสือที่ยังไม่คืน:
  http://example.org/lib#book1
  http://example.org/lib#book3


## Ontology Design Pattern: AgentRole

แทนที่จะสร้าง class `Borrower` แข็ง ๆ ให้ reify **บทบาท** ออกมาต่างหาก เพื่อให้คนคนเดียวกันเล่นได้หลายบทบาทพร้อมกัน

In [4]:
pattern_ttl = """
@prefix ex: <http://example.org/lib#> .
ex:alice a ex:Person .
ex:alice ex:hasRole ex:role1 .
ex:role1 a ex:BorrowerRole ; ex:roleOf ex:alice ; ex:forLoan ex:loan001 .
"""
g2 = Graph().parse(data=pattern_ttl, format="turtle")
print(f"{len(g2)} triples -- แลกความง่ายกับความยืดหยุ่น")

5 triples -- แลกความง่ายกับความยืดหยุ่น


## Anti-pattern: Class as Instance (ตรวจจับด้วยโค้ด)

IRI เดียวกันถูกใช้เป็นทั้ง `rdf:type owl:Class` และเป็น object ของ property ธรรมดา — สัญญาณของการสับสนระหว่าง class กับ instance

In [5]:
bad_ttl = """
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix ex:  <http://example.org/lib#> .

ex:Fiction a owl:Class .
ex:book1 ex:genre ex:Fiction .
"""
g3 = Graph().parse(data=bad_ttl, format="turtle")

classes = set(g3.subjects(RDF.type, OWL.Class))
used_as_object = {o for _, p, o in g3 if p != RDF.type}
suspicious = classes & used_as_object
print("Class ที่ถูกใช้เป็น instance ด้วย (น่าสงสัย):", suspicious)

Class ที่ถูกใช้เป็น instance ด้วย (น่าสงสัย): {rdflib.term.URIRef('http://example.org/lib#Fiction')}


## Structural Metrics — วัดคุณภาพเชิงโครงสร้าง

In [6]:
metrics_ttl = """
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix ex:   <http://example.org/lib#> .

ex:Item a owl:Class .
ex:Book a owl:Class ; rdfs:subClassOf ex:Item .
ex:DVD  a owl:Class ; rdfs:subClassOf ex:Item .
ex:hasTitle a owl:DatatypeProperty .
ex:borrowedBy a owl:ObjectProperty .
"""
g4 = Graph().parse(data=metrics_ttl, format="turtle")

classes = {c for c in g4.subjects(RDF.type, OWL.Class) if isinstance(c, URIRef)}
props = {p for p in g4.subjects(RDF.type, OWL.ObjectProperty)} | \
        {p for p in g4.subjects(RDF.type, OWL.DatatypeProperty)}

print(f"จำนวน class    : {len(classes)}")
print(f"จำนวน property : {len(props)}")
print(f"class/property ratio : {len(classes)/len(props):.2f}")

จำนวน class    : 3
จำนวน property : 2
class/property ratio : 1.50


## 🎯 สรุป

- เริ่มออกแบบจาก **Competency Question** แล้วแปลงเป็น SPARQL ไว้เป็นแบบทดสอบ
- **AgentRole pattern** แยก "บทบาท" ออกจาก "ตัวตน" เพื่อความยืดหยุ่น (แลกกับ triple ที่มากขึ้น)
- ตรวจ **anti-pattern** ได้ด้วยโค้ดง่าย ๆ เช่น หา IRI ที่ถูกใช้เป็นทั้ง class และ instance
- **Structural metrics** (จำนวน class/property, ratio) ช่วยประเมินคุณภาพเชิงปริมาณ

ลองเพิ่ม CQ ข้อใหม่ เช่น *"ใครยืมหนังสือเกิน 2 เล่ม"* แล้วเขียน SPARQL ตอบดู!

---

## จบบทที่ 10 🎉

ไปต่อบทถัดไป: **บทที่ 11 — SWRL**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*